# Lab: Prepare Data for Analyzing the Relationship Between Population and World Cup Performance

In this lab, we will:

- Load JSON and CSV data
- Explore nested data structures
- Extract all 2018 World Cup matches
- Identify unique participating teams
- Count each team's wins
- Filter and normalize population data
- Convert population values to integers
- Combine World Cup performance and population data into one nested dictionary

## Imports

In [ ]:
import json
import csv

## Step 1: Load the World Cup and Population Data

In [ ]:
with open("data/world_cup_2018.json", encoding="utf8") as world_cup_file:
    world_cup_data = json.load(world_cup_file)

with open("data/country_populations.csv") as population_file:
    population_data = list(csv.DictReader(population_file))

In [ ]:
assert type(world_cup_data) == dict
assert list(world_cup_data.keys()) == ['name', 'rounds']

In [ ]:
assert type(population_data) == list
assert type(population_data[0]) == dict

## Step 2: Explore the Structure of the World Cup JSON Data

In [ ]:
highest_lvl_keys = world_cup_data.keys()
print(highest_lvl_keys)

In [ ]:
print(world_cup_data["name"])
print(world_cup_data["rounds"][0])

In [ ]:
rounds = world_cup_data["rounds"]

## Step 3: Extract All Matches

In [ ]:
matches = []
for round_ in rounds:
    round_matches = round_["matches"]
    matches.extend(round_matches)
matches[0]

In [ ]:
assert len(matches) == 64
assert type(matches[0]) == dict

## Step 4: Extract Unique Team Names

In [ ]:
teams_set = set()
for match in matches:
    teams_set.add(match["team1"]["name"])
    teams_set.add(match["team2"]["name"])
teams = sorted(list(teams_set))
teams

In [ ]:
assert type(teams) == list
assert len(teams) == 32
assert type(teams[0]) == str

## Step 5: Initialize `combined_data`

In [ ]:
combined_data = {team: {"wins": 0} for team in teams}
combined_data

In [ ]:
assert type(combined_data) == dict
assert type(list(combined_data.keys())[0]) == str
assert combined_data["Japan"] == {"wins": 0}

## Step 6: Write a Function to Find the Winner of a Match

In [ ]:
def find_winner(match):
    if match["score1"] > match["score2"]:
        return match["team1"]["name"]
    elif match["score2"] > match["score1"]:
        return match["team2"]["name"]
    else:
        return None

In [ ]:
assert find_winner(matches[0]) == "Russia"
assert find_winner(matches[1]) == "Uruguay"
assert find_winner(matches[2]) == None

## Step 7: Add Win Counts to `combined_data`

In [ ]:
for match in matches:
    winner = find_winner(match)
    if winner:
        combined_data[winner]["wins"] += 1
combined_data

## Step 8: Filter Population Data to 2018 World Cup Teams

In [ ]:
population_data_filtered = []
for record in population_data:
    if record["Country Name"] in teams and record["Year"] == "2018":
        population_data_filtered.append(record)
len(population_data_filtered)

At this stage, the filtered list should contain 27 records because some country names differ between the two datasets.

## Step 9: Normalize Country Names and Rebuild the Filtered Population Data

In [ ]:
def normalize_location(country_name):
    name_sub_dict = {
        "Russian Federation": "Russia",
        "Egypt, Arab Rep.": "Egypt",
        "Iran, Islamic Rep.": "Iran",
        "Korea, Rep.": "South Korea",
        "United Kingdom": "England"
    }
    return name_sub_dict.get(country_name, country_name)

print(normalize_location("Russian Federation"))
print(normalize_location("Argentina"))

In [ ]:
population_data_filtered = []
for record in population_data:
    country = normalize_location(record["Country Name"])
    if country in teams and record["Year"] == "2018":
        record["Country Name"] = country
        population_data_filtered.append(record)
len(population_data_filtered)

In [ ]:
assert len(population_data_filtered) == 32

## Step 10: Convert Population Values from Strings to Integers

In [ ]:
for record in population_data_filtered:
    record["Value"] = int(record["Value"])
population_data_filtered[-1]

In [ ]:
assert type(population_data_filtered[-1]["Value"]) == int

## Step 11: Add Population Data to `combined_data`

In [ ]:
for record in population_data_filtered:
    country = record["Country Name"]
    population = record["Value"]
    combined_data[country]["population"] = population
combined_data

In [ ]:
assert type(combined_data["Uruguay"]) == dict
assert type(combined_data["Uruguay"]["population"]) == int

## Final Result

`combined_data` now contains one entry for each of the 32 countries that participated in the 2018 FIFA World Cup, with wins and 2018 population.